# 🛠️ Chapter 02b: Hands-On — Tokenization & Embeddings

---

## See The Magic Under The Hood! 🔍

In the concept notebook, we learned HOW transformers work — attention, embeddings, training pipeline. Now let's **play with these concepts in code.**

By the end of this notebook, you'll have:
- ✅ Tokenized text and seen how LLMs break words apart
- ✅ Generated real vector embeddings with OpenAI
- ✅ Built a semantic search engine in 20 lines of code
- ✅ Visualized embeddings in 2D to SEE meaning clusters

Let's go! 🚀

---

## 📦 Setup: Install & Import

Run this cell first:

In [ ]:
# Run this ONCE to install all required packages
# After running, you can comment it out

# !pip install openai langchain-openai langchain-core python-dotenv tiktoken numpy scikit-learn matplotlib

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✅ API key loaded! (starts with: {api_key[:8]}...)")
else:
    print("❌ No API key found! Create a .env file with OPENAI_API_KEY=sk-your-key")

---

## 🧪 Exercise 1: Tokenization Deep Dive

Remember from the concept notebook: LLMs don't see words — they see **tokens**.

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    TOKENIZATION RECAP                                 ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Human sees:  "Hello, how are you today?"                             ║
║                                                                       ║
║  LLM sees:    ["Hello", ",", " how", " are", " you", " today", "?"]  ║
║               (each piece is a TOKEN with a unique ID number)        ║
║                                                                       ║
║  The process: Text → Tokenizer → Token IDs → Model                  ║
║                                                                       ║
║  Why not just use words?                                              ║
║  • Handles ANY text (even gibberish, code, emojis)                   ║
║  • Fixed vocabulary size (~100K tokens)                               ║
║  • Common words = 1 token (efficient!)                                ║
║  • Rare words = multiple tokens (still works!)                       ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

Let's see this in action:

In [ ]:
import tiktoken

# Load the tokenizer used by GPT-4o
enc = tiktoken.encoding_for_model("gpt-4o")

# Tokenize different types of text
texts = [
    "Hello world",
    "I love programming in Python",
    "chatgpt",                        # A single made-up word
    "supercalifragilisticexpialidocious",  # Very long rare word
    "The capital of France is Paris",
    "def binary_search(arr, target):", # Code!
    "🚀🤖🎉",                        # Emojis
    "मैं Python सीख रहा हूं",          # Hindi
    "SELECT * FROM users WHERE age > 25", # SQL
]

print("TEXT → TOKENS")
print("=" * 70)

for text in texts:
    tokens = enc.encode(text)
    token_strings = [enc.decode([t]) for t in tokens]
    
    print(f"\n📝 \"{text}\"")
    print(f"   Token count: {len(tokens)}")
    print(f"   Token IDs:   {tokens}")
    print(f"   Broken into: {token_strings}")

### 🔍 Key Observations

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Common words → 1 token: "Hello", "the", "is"                  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 "chatgpt" → 3 tokens: ["chat", "g", "pt"]                    │
│     Even though it's one "word", the tokenizer splits it!         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Emojis → 2-3 tokens each — expensive!                         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Code keywords → usually 1 token ("def", "SELECT", "FROM")     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Non-English text → way more tokens per word!                   │
│     Hindi/Chinese/Arabic can cost 2-4x more than English           │
└─────────────────────────────────────────────────────────────────────┘
```

### 🧮 Token Cost Calculator

Let's build a tool that estimates how much any text costs to process:

In [ ]:
def count_and_cost(text, model="gpt-4o-mini"):
    """Count tokens and estimate cost for input text."""
    # Pricing per million tokens (2026 approximate)
    pricing = {
        "gpt-4o-mini":     {"input": 0.15,  "output": 0.60},
        "gpt-4o":          {"input": 2.50,  "output": 10.00},
        "claude-4-sonnet": {"input": 3.00,  "output": 15.00},
    }
    
    enc = tiktoken.encoding_for_model("gpt-4o")
    tokens = enc.encode(text)
    n_tokens = len(tokens)
    
    # Estimate: output is roughly 2x input for a typical Q&A
    estimated_output = n_tokens * 2
    
    prices = pricing[model]
    input_cost = (n_tokens / 1_000_000) * prices["input"]
    output_cost = (estimated_output / 1_000_000) * prices["output"]
    total = input_cost + output_cost
    
    return {
        "model": model,
        "input_tokens": n_tokens,
        "est_output_tokens": estimated_output,
        "cost_per_call": total,
        "calls_per_dollar": int(1 / total) if total > 0 else float('inf')
    }

# Compare costs across models
test_prompt = "Explain the transformer architecture and self-attention mechanism in detail."

print(f"📝 Prompt: \"{test_prompt}\"")
print("=" * 60)

for model in ["gpt-4o-mini", "gpt-4o", "claude-4-sonnet"]:
    result = count_and_cost(test_prompt, model)
    print(f"\n🤖 {result['model']}")
    print(f"   Input tokens:  {result['input_tokens']}")
    print(f"   Cost per call: ${result['cost_per_call']:.6f}")
    print(f"   💡 $1 gets you ~{result['calls_per_dollar']:,} calls!")

### 🤔 Quick Check

**Question:** You're building a chatbot for a hospital that handles 10,000 queries/day. Each query averages 100 input tokens and 200 output tokens. Should you use GPT-4o or GPT-4o-mini?

<details>
<summary>👆 Click to see answer</summary>

**GPT-4o-mini** for the vast majority of queries!

- GPT-4o-mini: (100 × $0.15 + 200 × $0.60) / 1M × 10,000 = **$0.0135/day** 
- GPT-4o: (100 × $2.50 + 200 × $10.00) / 1M × 10,000 = **$0.225/day**

GPT-4o costs ~17x more. For a hospital, you'd likely use mini for simple FAQs and route complex medical questions to GPT-4o. This is called **model routing** — a real production pattern!

</details>

---

## 🧪 Exercise 2: Generate Real Embeddings

Now for the **most mind-blowing** concept in all of GenAI: **embeddings**.

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    WHAT ARE EMBEDDINGS?                                ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  An embedding converts TEXT into a list of NUMBERS (a vector).       ║
║                                                                       ║
║  "I love Python" → [0.023, -0.041, 0.087, ..., 0.012]              ║
║                      (1536 numbers for text-embedding-3-small)       ║
║                                                                       ║
║  WHY? Because computers are great with numbers, not words!           ║
║                                                                       ║
║  THE MAGIC:                                                           ║
║  • Similar meanings → NEARBY vectors                                 ║
║  • Different meanings → FAR APART vectors                            ║
║                                                                       ║
║  "happy" ←→ "joyful"     = CLOSE (similar meaning)                  ║
║  "happy" ←→ "refrigerator" = FAR (unrelated)                        ║
║                                                                       ║
║  This is the FOUNDATION of:                                          ║
║  • Semantic search (find similar documents)                           ║
║  • RAG (Retrieval-Augmented Generation)                              ║
║  • Recommendation systems                                             ║
║  • Clustering and classification                                      ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
from openai import OpenAI

client = OpenAI()

# Generate an embedding for a single sentence
text = "I love programming in Python"

response = client.embeddings.create(
    model="text-embedding-3-small",  # OpenAI's embedding model
    input=text
)

# Extract the embedding vector
embedding = response.data[0].embedding

print(f"📝 Text: \"{text}\"")
print(f"📏 Embedding dimensions: {len(embedding)}")
print(f"🔢 First 10 values: {embedding[:10]}")
print(f"📊 Tokens used: {response.usage.total_tokens}")

### 💡 That's It! One Line and You Get a Vector!

The sentence "I love programming in Python" is now a list of **1536 numbers** that capture its **meaning** — not just its words.

Now let's prove that similar sentences have similar embeddings:

In [ ]:
import numpy as np

def get_embedding(text):
    """Get the embedding vector for a piece of text."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a, b):
    """Calculate how similar two vectors are (0 to 1)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test pairs of sentences
pairs = [
    ("I love Python programming", "Python is my favorite coding language"),
    ("I love Python programming", "The weather is nice today"),
    ("The cat sat on the mat", "A kitten was resting on the rug"),
    ("The cat sat on the mat", "Stock prices fell sharply"),
    ("Machine learning is a subset of AI", "AI includes deep learning and NLP"),
    ("Machine learning is a subset of AI", "I had pizza for dinner"),
]

print("SEMANTIC SIMILARITY TEST")
print("=" * 70)

for text1, text2 in pairs:
    emb1 = get_embedding(text1)
    emb2 = get_embedding(text2)
    similarity = cosine_similarity(emb1, emb2)
    
    # Visual bar
    bar = "█" * int(similarity * 30) + "░" * (30 - int(similarity * 30))
    
    print(f"\n📝 \"{text1}\"")
    print(f"📝 \"{text2}\"")
    print(f"   Similarity: {similarity:.4f} [{bar}]")

### 🔍 What Did You Notice?

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 SIMILAR meaning → HIGH similarity (0.7-0.9+)                   │
│     "Python programming" vs "favorite coding language" = HIGH      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 DIFFERENT meaning → LOW similarity (0.1-0.4)                   │
│     "Python programming" vs "weather is nice" = LOW               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 It understands MEANING, not just words!                        │
│     "cat sat on the mat" ≈ "kitten resting on the rug"            │
│     Different words, same meaning → HIGH similarity               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 This is EXACTLY how RAG works!                                 │
│     Your question gets embedded → find similar document chunks    │
│     → feed those chunks to the LLM → get an accurate answer      │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercise 3: Build a Semantic Search Engine! 🔍

Now let's build something REAL — a semantic search engine that finds relevant documents based on **meaning**, not just keywords.

```
╔═══════════════════════════════════════════════════════════════════════╗
║              SEMANTIC SEARCH vs KEYWORD SEARCH                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Query: "How to make my code run faster?"                             ║
║                                                                       ║
║  🔤 KEYWORD SEARCH:                                                  ║
║     Looks for exact words: "code", "run", "faster"                   ║
║     ❌ Misses: "Performance optimization techniques"                 ║
║     ❌ Misses: "Reducing algorithm complexity"                       ║
║                                                                       ║
║  🧠 SEMANTIC SEARCH (embeddings):                                    ║
║     Understands MEANING of the query                                 ║
║     ✅ Finds: "Performance optimization techniques"                  ║
║     ✅ Finds: "Reducing algorithm complexity"                        ║
║     ✅ Finds: "Tips for efficient Python coding"                     ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
import numpy as np

# Our "knowledge base" — imagine these are chunks from documents
documents = [
    "Python lists are ordered, mutable sequences that can hold any type of data.",
    "LangChain is a framework for building applications with large language models.",
    "RAG stands for Retrieval-Augmented Generation, combining search with LLM generation.",
    "Binary search requires the array to be sorted and runs in O(log n) time.",
    "Embeddings convert text into numerical vectors that capture semantic meaning.",
    "Vector databases like ChromaDB and FAISS store and search embedding vectors efficiently.",
    "Fine-tuning adapts a pre-trained model to perform better on specific tasks.",
    "Temperature controls the randomness of LLM outputs during text generation.",
    "Prompt engineering is the art of crafting effective inputs for language models.",
    "Self-attention allows transformers to weigh the importance of different words in context.",
    "AI agents can use tools, make decisions, and execute multi-step plans autonomously.",
    "Cosine similarity measures the angle between two vectors to determine how similar they are.",
]

# Step 1: Embed all documents (in real apps, you'd store these in a vector DB)
print("📦 Embedding all documents...")
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=documents
)
doc_embeddings = np.array([item.embedding for item in response.data])
print(f"✅ Embedded {len(documents)} documents (shape: {doc_embeddings.shape})")


def semantic_search(query, top_k=3):
    """Search documents by meaning, not keywords!"""
    # Step 2: Embed the query
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    query_embedding = np.array(query_response.data[0].embedding)
    
    # Step 3: Calculate similarity with ALL documents
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = np.dot(query_embedding, doc_emb) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_emb)
        )
        similarities.append((i, sim))
    
    # Step 4: Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Step 5: Return top results
    print(f"\n🔍 Query: \"{query}\"")
    print("=" * 60)
    for rank, (idx, score) in enumerate(similarities[:top_k], 1):
        bar = "█" * int(score * 30) + "░" * (30 - int(score * 30))
        print(f"\n  #{rank} (score: {score:.4f}) [{bar}]")
        print(f"     📄 {documents[idx]}")


# Let's search!
semantic_search("How do I store and find similar texts?")

In [ ]:
# Try more queries — notice how it understands MEANING!
semantic_search("What frameworks help me build AI apps?")

In [ ]:
# This is the magic — different words, same meaning!
semantic_search("How to make an LLM less random and more predictable?")

In [ ]:
# Even questions about concepts it wasn't directly trained on!
semantic_search("How does the transformer model understand word relationships?")

### 💡 This Is Basically RAG!

What we just built is the **core of RAG** (Retrieval-Augmented Generation):

```
┌─────────────────────────────────────────────────────────────────────┐
│  WHAT WE BUILT:                                                     │
│                                                                     │
│  1. Documents → Embed → Store (our numpy array)                    │
│  2. Query → Embed → Compare to all documents                       │
│  3. Return most similar documents                                   │
│                                                                     │
│  IN REAL RAG (Chapter 5), you'd add:                               │
│  4. Feed the top documents to an LLM as context                    │
│  5. LLM generates an answer USING those documents                  │
│                                                                     │
│  That's it! That's the whole idea! 🎉                               │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercise 4: Visualize Embeddings in 2D 📊

Embeddings live in 1536-dimensional space — we can't see that. But we can **project them down to 2D** using a technique called PCA (Principal Component Analysis) and literally **see meaning clusters**.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Let's embed sentences from different TOPICS
sentences = [
    # Programming
    "Python is a popular programming language",
    "JavaScript is used for web development",
    "Functions help organize code into reusable blocks",
    "Object-oriented programming uses classes and objects",
    
    # Food
    "Pizza is a delicious Italian dish",
    "Sushi is a traditional Japanese food",
    "Cooking requires fresh ingredients and patience",
    "Biryani is a flavorful rice dish from India",
    
    # Sports
    "Cricket is the most popular sport in India",
    "Football is played by billions worldwide",
    "Athletes train hard to compete in the Olympics",
    "Basketball requires speed, agility, and teamwork",
    
    # AI/ML
    "Neural networks are inspired by the human brain",
    "Large language models generate text using transformers",
    "Machine learning algorithms learn patterns from data",
    "Deep learning uses multiple layers to extract features",
]

categories = [
    "💻 Code", "💻 Code", "💻 Code", "💻 Code",
    "🍕 Food", "🍕 Food", "🍕 Food", "🍕 Food",
    "⚽ Sports", "⚽ Sports", "⚽ Sports", "⚽ Sports",
    "🤖 AI/ML", "🤖 AI/ML", "🤖 AI/ML", "🤖 AI/ML",
]

# Step 1: Get embeddings for all sentences
print("📦 Embedding sentences...")
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentences
)
embeddings = np.array([item.embedding for item in response.data])
print(f"✅ Got {len(embeddings)} embeddings of dimension {len(embeddings[0])}")

# Step 2: Reduce to 2D using PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)
print(f"📐 Reduced to 2D (explained variance: {sum(pca.explained_variance_ratio_):.2%})")

# Step 3: Plot!
fig, ax = plt.subplots(figsize=(12, 8))

colors = {"💻 Code": "#2196F3", "🍕 Food": "#FF5722", "⚽ Sports": "#4CAF50", "🤖 AI/ML": "#9C27B0"}

for i, (x, y) in enumerate(embeddings_2d):
    color = colors[categories[i]]
    ax.scatter(x, y, c=color, s=100, zorder=5)
    # Truncate long labels
    label = sentences[i][:40] + "..." if len(sentences[i]) > 40 else sentences[i]
    ax.annotate(label, (x, y), fontsize=7, ha='left', va='bottom',
                xytext=(5, 5), textcoords='offset points')

# Add legend
for cat, color in colors.items():
    ax.scatter([], [], c=color, s=100, label=cat)
ax.legend(fontsize=12, loc='upper left')

ax.set_title("🧠 Embeddings in 2D — Similar Topics Cluster Together!", fontsize=14)
ax.set_xlabel("PCA Dimension 1")
ax.set_ylabel("PCA Dimension 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🔍 What You Should See:

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    THE CLUSTERING EFFECT                              ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  💻 Code sentences cluster TOGETHER in one area                      ║
║  🍕 Food sentences cluster TOGETHER in another area                  ║
║  ⚽ Sports sentences form their own cluster                          ║
║  🤖 AI/ML sentences group near each other                            ║
║                                                                       ║
║  🤯 MIND-BLOWING: Notice that 💻 Code and 🤖 AI/ML are             ║
║     closer to each other than to 🍕 Food or ⚽ Sports!              ║
║     Because programming and AI are semantically related!             ║
║                                                                       ║
║  This is EXACTLY how vector databases work:                          ║
║  Store embeddings → Query comes in → Find nearest neighbors         ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🧪 Bonus: The Famous "King - Man + Woman = Queen" Test! 👑

One of the most famous embedding properties — let's test it:

In [ ]:
# The famous word analogy test!
# king - man + woman should be close to "queen"

words = ["king", "man", "woman", "queen", "prince", "princess", "boy", "girl"]

# Get embeddings for all words
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=words
)

word_embs = {word: np.array(item.embedding) 
             for word, item in zip(words, response.data)}

# king - man + woman = ???
result_vector = word_embs["king"] - word_embs["man"] + word_embs["woman"]

# Find which word is closest to this result
print("🧪 Testing: king - man + woman = ???")
print("=" * 50)

for word, emb in word_embs.items():
    sim = np.dot(result_vector, emb) / (np.linalg.norm(result_vector) * np.linalg.norm(emb))
    bar = "█" * int(sim * 20) + "░" * (20 - int(sim * 20))
    marker = " 👑" if word == "queen" else ""
    print(f"  {word:10s}: {sim:.4f} [{bar}]{marker}")

print("\n💡 'queen' should have one of the highest scores!")

---

## ✍️ Practice: Build a FAQ Matcher

**Challenge:** Build a function that matches a user's question to the most relevant FAQ answer.

This is a real-world use case — many customer support systems work exactly like this!

In [ ]:
# FAQ database
faqs = [
    {"q": "How do I reset my password?", "a": "Go to Settings → Security → Reset Password."},
    {"q": "What payment methods do you accept?", "a": "We accept UPI, credit cards, debit cards, and net banking."},
    {"q": "How do I cancel my subscription?", "a": "Go to Settings → Subscription → Cancel Plan."},
    {"q": "What is your refund policy?", "a": "Full refund within 7 days, 50% refund within 30 days."},
    {"q": "How do I contact support?", "a": "Email support@example.com or call 1800-123-4567."},
    {"q": "Can I upgrade my plan?", "a": "Yes! Go to Settings → Subscription → Upgrade."},
]

def build_faq_matcher(faqs):
    """
    YOUR CODE HERE
    
    1. Extract all questions from the FAQ list
    2. Embed them all using OpenAI
    3. Return the embeddings array
    """
    # TODO: Get embeddings for all FAQ questions
    # TODO: Return numpy array of embeddings
    pass

def find_best_faq(user_question, faq_embeddings, faqs, top_k=2):
    """
    YOUR CODE HERE
    
    1. Embed the user's question
    2. Compare against all FAQ embeddings
    3. Return the top_k most similar FAQs
    """
    # TODO: Embed the user question
    # TODO: Calculate cosine similarity with each FAQ
    # TODO: Return top matches
    pass

# Test:
# faq_embs = build_faq_matcher(faqs)
# find_best_faq("I forgot my password and can't log in", faq_embs, faqs)
# find_best_faq("I want my money back", faq_embs, faqs)
# find_best_faq("How can I talk to a human?", faq_embs, faqs)

<details>
<summary>💡 Click to see solution</summary>

```python
def build_faq_matcher(faqs):
    questions = [faq["q"] for faq in faqs]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=questions
    )
    return np.array([item.embedding for item in response.data])

def find_best_faq(user_question, faq_embeddings, faqs, top_k=2):
    # Embed user question
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=user_question
    )
    query_emb = np.array(response.data[0].embedding)
    
    # Calculate similarities
    similarities = []
    for i, faq_emb in enumerate(faq_embeddings):
        sim = np.dot(query_emb, faq_emb) / (
            np.linalg.norm(query_emb) * np.linalg.norm(faq_emb))
        similarities.append((i, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n🔍 User asked: \"{user_question}\"")
    for rank, (idx, score) in enumerate(similarities[:top_k], 1):
        print(f"  #{rank} (match: {score:.3f})")
        print(f"     Q: {faqs[idx]['q']}")
        print(f"     A: {faqs[idx]['a']}")
```

</details>

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║            INTERVIEW QUESTIONS FOR THIS MODULE                       ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: Explain the Transformer architecture in simple terms.            ║
║  A: The Transformer processes all tokens in parallel using          ║
║     self-attention (each token looks at every other token to        ║
║     understand context). It has an encoder (understands input)      ║
║     and decoder (generates output). Key innovation: attention       ║
║     mechanism replaced sequential processing (RNNs), enabling      ║
║     massive parallelization and much better long-range context.    ║
║                                                                       ║
║  Q: What is self-attention? Why is it important?                    ║
║  A: Self-attention lets each token compute a weighted sum of all    ║
║     other tokens based on relevance. E.g., in "The bank by the     ║
║     river", the word "bank" attends strongly to "river" (not       ║
║     "financial"). This captures context and resolves ambiguity.    ║
║     Without it, models couldn't understand long-range dependencies.║
║                                                                       ║
║  Q: What are embeddings? How do they capture meaning?               ║
║  A: Embeddings are dense vectors (lists of numbers) that represent ║
║     text in a high-dimensional space where SIMILAR meanings are    ║
║     CLOSE together. Generated by neural networks trained on        ║
║     massive text corpora. Used for: semantic search, RAG,          ║
║     classification, clustering. Cosine similarity measures how     ║
║     close two embedding vectors are.                               ║
║                                                                       ║
║  Q: Why do LLMs hallucinate? How do you mitigate it?               ║
║  A: LLMs are probabilistic text generators — they produce          ║
║     statistically likely continuations, which may be wrong.        ║
║     Mitigations:                                                    ║
║     • RAG (ground responses in retrieved facts)                    ║
║     • Temperature=0 (reduce randomness)                             ║
║     • Tool use (let LLM call APIs for facts)                       ║
║     • Guardrails (validate outputs before showing user)            ║
║     • Fine-tuning on domain data                                    ║
║                                                                       ║
║  Q: What is RLHF?                                                    ║
║  A: Reinforcement Learning from Human Feedback. After pre-training,║
║     the model generates multiple responses → humans rank them      ║
║     → a reward model is trained → the LLM is fine-tuned using     ║
║     this reward signal. This makes outputs more helpful, safe,     ║
║     and aligned with human preferences. Key to ChatGPT's success. ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Quick Quiz

---

### Question 1:
What does a text embedding represent?

- A) A compressed version of the text
- B) A numerical vector that captures the semantic meaning of the text
- C) A list of keywords extracted from the text
- D) The text converted to binary

<details><summary>Answer</summary>

**B) A numerical vector that captures the semantic meaning of the text**

Embeddings map text to dense vectors where similar meanings are close together in vector space. This enables semantic search, clustering, and similarity comparison.

</details>

---

### Question 2:
Two sentences have a cosine similarity of 0.95. What does this mean?

- A) They share 95% of the same words
- B) They have very similar semantic meaning
- C) One is 95% as long as the other
- D) They were generated by the same model

<details><summary>Answer</summary>

**B) They have very similar semantic meaning**

Cosine similarity measures the angle between vectors. 1.0 = identical direction (same meaning), 0.0 = perpendicular (unrelated). 0.95 means very semantically similar, even if the WORDS are completely different!

</details>

---

### Question 3:
Why does Hindi text use more tokens than English text of the same meaning?

- A) Hindi has longer words
- B) The tokenizer was trained primarily on English data, so Hindi characters are less efficiently encoded
- C) Hindi has more grammar rules
- D) Hindi text is always longer

<details><summary>Answer</summary>

**B) The tokenizer was trained primarily on English data, so Hindi characters are less efficiently encoded**

BPE tokenizers learn common patterns from training data. English patterns (common words, subwords) get single tokens. Hindi characters are rarer in training data, so they get split into more tokens. This means non-English text costs more per API call!

</details>

---

### Question 4:
What is the core pipeline of RAG?

- A) Train → Fine-tune → Deploy
- B) Embed documents → Store → Embed query → Retrieve similar → Generate with LLM
- C) Prompt → Response → Evaluate
- D) Tokenize → Encode → Decode → Detokenize

<details><summary>Answer</summary>

**B) Embed documents → Store → Embed query → Retrieve similar → Generate with LLM**

RAG = Retrieval-Augmented Generation. You embed and store your documents, then at query time, embed the question, find the most similar document chunks, and feed them as context to the LLM. We'll build full RAG pipelines in Chapter 5!

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Tokens ≠ words. BPE splits text into subword chunks.          │
│     Non-English and rare words use MORE tokens.                    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Embeddings turn text into numbers (vectors) that capture      │
│     MEANING, not just characters.                                  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Similar meaning → nearby vectors (high cosine similarity)     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Semantic search: embed query → compare to doc embeddings      │
│     → return most similar. This is the core of RAG!               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 text-embedding-3-small → 1536 dimensions, cheap, fast         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 PCA can reduce dimensions for visualization — topics cluster! │
├─────────────────────────────────────────────────────────────────────┤
│  📌 "king - man + woman ≈ queen" — embeddings encode relationships│
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Prompt Engineering!

Now that you understand the mechanics of LLMs (how they tokenize, embed, and generate), it's time to learn the art of **talking to them effectively** — prompt engineering!

---

### 🎉 Incredible Work!
You've tokenized text, generated embeddings, built a semantic search engine, AND visualized meaning in 2D. You're thinking like an AI engineer! 🚀